In [15]:
import dotenv
from web3 import Web3
import json
import os

dotenv.load_dotenv()

True

In [16]:
# Load config
with open("config.json") as f:
    config = json.load(f)

w3 = Web3(Web3.HTTPProvider(config["blockchain"]["rpc_url"]))
print("Connected:", w3.is_connected())

stablecoin_address = config["blockchain"]["stablecoin_address"]
p2p_market_address = config["blockchain"]["p2p_market_address"]

# Minimal ERC-20 ABI — only what we need for checking allowances
erc20_abi = [
    {
        "constant": True,
        "inputs": [
            {"name": "owner", "type": "address"},
            {"name": "spender", "type": "address"}
        ],
        "name": "allowance",
        "outputs": [{"name": "", "type": "uint256"}],
        "type": "function"
    },
    {
        "constant": True,
        "inputs": [{"name": "account", "type": "address"}],
        "name": "balanceOf",
        "outputs": [{"name": "", "type": "uint256"}],
        "type": "function"
    },
    {
        "constant": False,
        "inputs": [
            {"name": "spender", "type": "address"},
            {"name": "amount", "type": "uint256"}
        ],
        "name": "approve",
        "outputs": [{"name": "", "type": "bool"}],
        "type": "function"
    }
]

stablecoin = w3.eth.contract(address=Web3.to_checksum_address(stablecoin_address), abi=erc20_abi)

Connected: True


In [17]:
account_Deploy = w3.eth.account.from_key(os.environ["DEPLOYER_PRIVATE_KEY"])
print("Connected, chain id:", w3.eth.chain_id)
print("Account:", account_Deploy.address)
print("Balance (ETH):", w3.from_wei(w3.eth.get_balance(account_Deploy.address), "ether"))

Connected, chain id: 11155111
Account: 0x2C4b47689a05f0653637a97a8Db762b764f1653c
Balance (ETH): 0.307954919985936293


In [18]:
account_trigger = w3.eth.account.from_key(os.environ["TRIGGER_PRIVATE_KEY"])
print("Connected, chain id:", w3.eth.chain_id)
print("Account:", account_trigger.address)
print("Balance (ETH):", w3.from_wei(w3.eth.get_balance(account_trigger.address), "ether"))

Connected, chain id: 11155111
Account: 0x4022d2250AB1E76d3fcbCcf18d39656f4559b83c
Balance (ETH): 0.163119127731955958


In [19]:
def load_contract(abi_path: str, address: str):
    with open(abi_path) as f:
        artifact = json.load(f)
    abi = artifact["abi"] if "abi" in artifact else artifact
    return w3.eth.contract(address=Web3.to_checksum_address(address), abi=abi)

bc = config["blockchain"]

oracle_storage = load_contract("abi/OracleStorage.json", bc["oracle_storage_address"])
p2p_market = load_contract("abi/P2PEnergyMarket.json", bc["p2p_market_address"])

oracle_storage.address, p2p_market.address

('0x61769c1C299495194D7Df49030019e613d13a88D',
 '0xB5D7DE4841985feA6a234256B5Adfa4AF1725bF7')

In [20]:
oracle_storage.functions.authorizeOracle(os.getenv("ORACLE_PRIVATE_KEY"))

<Function authorizeOracle(address) bound to ('bb0a0d5667ab20e5f60c80e65afef6188f90a92cf72e689aca335b2bf9dc57b7',)>

In [21]:
for household in config["households"]:
    p2p_market.functions.registerHousehold(household["address"])

In [27]:
for household in config["households"]:
    addr = Web3.to_checksum_address(household["address"])
    allowance = stablecoin.functions.allowance(addr, p2p_market_address).call()
    balance = stablecoin.functions.balanceOf(addr).call()
    print(f"{household.get('name', addr)}: allowance={allowance}, balance={balance}")

0x2C4b47689a05f0653637a97a8Db762b764f1653c: allowance=115792089237316195423570985008687907853269984665640564039457584007913129639935, balance=65301800
0xF49153d700AD86CA224f7B2064F541278FE1c320: allowance=115792089237316195423570985008687907853269984665640564039457584007913129639935, balance=5068600
0xC416DDdD40c28eAfE48275f74B4E4Bdee4E3e75b: allowance=115792089237316195423570985008687907853269984665640564039457584007913129639935, balance=29629600


In [23]:
def send_tx(contract_function, signer=None, gas: int = 200_000):
    signer = signer   # defaults to the notebook's main account
    nonce = w3.eth.get_transaction_count(signer.address, "pending")
    tx = contract_function.build_transaction({
        "from": signer.address,
        "nonce": nonce,
        "gas": gas,
        "chainId": w3.eth.chain_id,
    })
    signed = signer.sign_transaction(tx)
    tx_hash = w3.eth.send_raw_transaction(signed.raw_transaction)
    receipt = w3.eth.wait_for_transaction_receipt(tx_hash)
    status = "OK" if receipt.status == 1 else "REVERTED"
    print(f"{status} | tx {tx_hash.hex()} | gasUsed {receipt.gasUsed}")
    return receipt

In [24]:
MAX_UINT256 = 2**256 - 1
ALREADY_APPROVED_THRESHOLD = 2**200  # treat anything this large as "already unlimited"

for i, household in enumerate(config["households"], start=1):
    label = household.get("name", household["id"])
    addr = Web3.to_checksum_address(household["address"])
    env_var = f"HOUSEHOLD{i}_PRIVATE_KEY"

    private_key = os.getenv(env_var)
    if not private_key:
        print(f"✗ {label}: {env_var} not set in .env, skipping")
        continue

    hh_account = w3.eth.account.from_key(private_key)
    if hh_account.address.lower() != addr.lower():
        print(f"✗ {label}: {env_var} resolves to {hh_account.address}, "
              f"but config.json has {addr} — skipping")
        continue

    current_allowance = stablecoin.functions.allowance(addr, p2p_market_address).call()
    if current_allowance >= ALREADY_APPROVED_THRESHOLD:
        print(f"✓ {label}: already approved ({current_allowance})")
        continue

    print(f"→ {label}: approving (current allowance {current_allowance})")
    send_tx(
        stablecoin.functions.approve(p2p_market_address, MAX_UINT256),
        signer=hh_account,
        gas=100_000,
    )

→ house_01: approving (current allowance 0)


OK | tx c46cbb56f66d575ebb13d2f22bb9825cada5702030f4e8a2256aa21b2c18f288 | gasUsed 51658
→ house_02: approving (current allowance 50)
OK | tx fca110a33ff41f351f4fc62c5fc4ae88cc05e1cc338bfe0c65a84dfefcd24f88 | gasUsed 34558
→ house_03: approving (current allowance 0)
OK | tx 1b3f4a85e53743c2c922f6ad80b791cee8b263f0bc6968d8563bcfa69ae0df45 | gasUsed 51658


In [25]:
for i, household in enumerate(config["households"], start=1):
    label = household.get("name", household["id"])
    addr = Web3.to_checksum_address(household["address"])
    print(addr)

0x2C4b47689a05f0653637a97a8Db762b764f1653c
0xF49153d700AD86CA224f7B2064F541278FE1c320
0xC416DDdD40c28eAfE48275f74B4E4Bdee4E3e75b
